# Handlers

> Route handlers and router initialization for the file browser.

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| export
from typing import Any, Callable, Optional

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import (
    FileBrowserConfig, FileBrowserCallbacks, SelectionMode
)
from cjm_fasthtml_file_browser.core.models import BrowserState
from cjm_fasthtml_file_browser.core.protocols import FileSystemProvider
from cjm_fasthtml_file_browser.components.browser import render_file_browser

## Navigation Handlers

In [ ]:
#| export
def _handle_navigate(
    provider: FileSystemProvider,           # File system provider
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path to navigate to
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle navigation to a new directory."""
    # Validate navigation if callback provided
    if callbacks and callbacks.validate_navigation:
        valid, error = callbacks.validate_navigation(path)
        if not valid:
            state = state_getter()
            return render_fn(state)
    
    # Normalize path
    normalized = provider.normalize_path(path)
    
    # Update state
    state = state_getter()
    state.current_path = normalized
    state_setter(state)
    
    # Call callback if provided
    if callbacks and callbacks.on_navigate:
        callbacks.on_navigate(normalized)
    
    return render_fn(state)

In [ ]:
#| export
def _handle_path_input(
    provider: FileSystemProvider,           # File system provider
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path input by user
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
    navigate_fn: Callable[[str], Any],      # Function to handle navigation
) -> Any:  # Rendered browser component
    """Handle direct path input."""
    # Validate path exists and is directory
    valid, error = provider.is_valid_path(path)
    if valid and provider.is_directory(path):
        return navigate_fn(path)
    else:
        # Invalid path - stay on current
        state = state_getter()
        return render_fn(state)

## Selection Handlers

In [ ]:
#| export
def _handle_select(
    config: FileBrowserConfig,              # Browser configuration
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    callbacks: Optional[FileBrowserCallbacks],  # Optional callbacks
    path: str,                              # Path to select/deselect
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle file selection."""
    # Validate selection if callback provided
    if callbacks and callbacks.validate_selection:
        valid, error = callbacks.validate_selection(path)
        if not valid:
            state = state_getter()
            return render_fn(state)
    
    state = state_getter()
    
    # Handle selection based on mode
    if config.selection_mode == SelectionMode.SINGLE:
        if state.selection.is_selected(path):
            state.selection.clear()
        else:
            state.selection.set_single(path)
    elif config.selection_mode == SelectionMode.MULTIPLE:
        state.selection.toggle(path)
        # Check max selections
        if config.max_selections and len(state.selection.selected_paths) > config.max_selections:
            # Remove oldest selection
            state.selection.selected_paths = state.selection.selected_paths[-config.max_selections:]
    
    state_setter(state)
    
    # Call callbacks
    if callbacks and callbacks.on_select:
        callbacks.on_select(path)
    if callbacks and callbacks.on_selection_change:
        callbacks.on_selection_change(state.selection.selected_paths)
    
    return render_fn(state)

## View and Sort Handlers

In [ ]:
#| export
def _handle_toggle_view(
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    view_mode: str,                         # New view mode ("list" or "grid")
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle view mode toggle."""
    state = state_getter()
    state.view_mode = view_mode
    state_setter(state)
    return render_fn(state)

In [ ]:
#| export
def _handle_change_sort(
    state_getter: Callable[[], BrowserState],  # Function to get current state
    state_setter: Callable[[BrowserState], None],  # Function to save state
    sort_by: str,                           # Column to sort by
    sort_descending: str,                   # Whether to sort descending ("true"/"false")
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle sort change."""
    state = state_getter()
    state.sort_by = sort_by
    state.sort_descending = sort_descending.lower() == "true"
    state_setter(state)
    return render_fn(state)

In [ ]:
#| export
def _handle_refresh(
    state_getter: Callable[[], BrowserState],  # Function to get current state
    render_fn: Callable[[BrowserState], Any],  # Function to render browser
) -> Any:  # Rendered browser component
    """Handle refresh (re-render with current state)."""
    state = state_getter()
    return render_fn(state)

## Router Initialization

The `init_router` function creates and configures an APIRouter with all file browser routes.
Route paths are automatically derived from function names.

In [ ]:
#| export
def init_router(
    config: FileBrowserConfig,                              # Browser configuration
    provider: FileSystemProvider,                           # File system provider
    state_getter: Callable[[], BrowserState],               # Function to get current state
    state_setter: Callable[[BrowserState], None],           # Function to save state
    route_prefix: str = "/browser",                         # Route prefix for all browser routes
    callbacks: Optional[FileBrowserCallbacks] = None,       # Optional callbacks
    home_path: Optional[str] = None,                        # Home directory (defaults to provider)
) -> APIRouter:  # Configured APIRouter with all file browser routes
    """
    Initialize and return an APIRouter with all file browser routes.
    
    Route paths are automatically derived from function names:
    - navigate -> {prefix}/navigate
    - select -> {prefix}/select
    - toggle_view -> {prefix}/toggle_view
    - change_sort -> {prefix}/change_sort
    - refresh -> {prefix}/refresh
    - path_input -> {prefix}/path_input
    
    Example:
    ```python
    from cjm_fasthtml_app_core.core.routing import register_routes
    
    router = init_router(
        config=config,
        provider=provider,
        state_getter=get_state,
        state_setter=set_state,
        route_prefix="/browser",
    )
    register_routes(app, router)
    ```
    """
    router = APIRouter(prefix=route_prefix)
    home = home_path or (provider.get_home_path() if hasattr(provider, 'get_home_path') else "/")
    
    def _render_browser(state: BrowserState) -> Any:
        """Helper to render browser with current state."""
        listing = provider.list_directory(state.current_path)
        return render_file_browser(
            listing=listing,
            config=config,
            state=state,
            navigate_url=navigate.to(),
            select_url=select.to(),
            toggle_view_url=toggle_view.to(),
            change_sort_url=change_sort.to(),
            refresh_url=refresh.to(),
            path_input_url=path_input.to(),
            home_path=home,
        )
    
    # -------------------------------------------------------------------------
    # Navigation Routes
    # -------------------------------------------------------------------------
    
    @router
    def navigate(path: str) -> Any:
        """Navigate to a new directory."""
        return _handle_navigate(
            provider=provider,
            state_getter=state_getter,
            state_setter=state_setter,
            callbacks=callbacks,
            path=path,
            render_fn=_render_browser,
        )
    
    @router
    def path_input(path: str) -> Any:
        """Handle direct path input."""
        return _handle_path_input(
            provider=provider,
            state_getter=state_getter,
            state_setter=state_setter,
            callbacks=callbacks,
            path=path,
            render_fn=_render_browser,
            navigate_fn=navigate,
        )
    
    @router
    def refresh() -> Any:
        """Refresh the current directory listing."""
        return _handle_refresh(
            state_getter=state_getter,
            render_fn=_render_browser,
        )
    
    # -------------------------------------------------------------------------
    # Selection Routes
    # -------------------------------------------------------------------------
    
    @router
    def select(path: str) -> Any:
        """Select or deselect a file/folder."""
        return _handle_select(
            config=config,
            state_getter=state_getter,
            state_setter=state_setter,
            callbacks=callbacks,
            path=path,
            render_fn=_render_browser,
        )
    
    # -------------------------------------------------------------------------
    # View and Sort Routes
    # -------------------------------------------------------------------------
    
    @router
    def toggle_view(view_mode: str) -> Any:
        """Toggle between list and grid view."""
        return _handle_toggle_view(
            state_getter=state_getter,
            state_setter=state_setter,
            view_mode=view_mode,
            render_fn=_render_browser,
        )
    
    @router
    def change_sort(sort_by: str, sort_descending: str = "false") -> Any:
        """Change the sort column and direction."""
        return _handle_change_sort(
            state_getter=state_getter,
            state_setter=state_setter,
            sort_by=sort_by,
            sort_descending=sort_descending,
            render_fn=_render_browser,
        )
    
    return router

In [ ]:
import tempfile
from pathlib import Path
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider

# Create test fixtures
with tempfile.TemporaryDirectory() as tmpdir:
    # Create test files
    (Path(tmpdir) / "file1.txt").write_text("content1")
    (Path(tmpdir) / "subdir").mkdir()
    
    # Set up browser
    config = FileBrowserConfig()
    provider = LocalFileSystemProvider()
    
    # Simple state storage
    _state = BrowserState(current_path=tmpdir)
    def get_state(): return _state
    def set_state(s): global _state; _state = s
    
    # Create router
    router = init_router(
        config=config,
        provider=provider,
        state_getter=get_state,
        state_setter=set_state,
        route_prefix="/browser",
    )
    
    # Verify router was created
    assert router is not None
    assert router.prefix == "/browser"

print("Router initialization tests passed!")

Router initialization tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()